Importing the libraries and setup


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Duke perdorur paisjen : {device}")


IMG_SIZE = 128
base_dir = "SOD_Project"
os.makedirs(base_dir + "/data/images", exist_ok=True)
os.makedirs(base_dir + "/data/masks", exist_ok=True)

Duke perdorur paisjen : cpu


Create project structure


In [ ]:
import os

base_dir = "SOD_Project"

os.makedirs(base_dir + "/data/images", exist_ok=True)
os.makedirs(base_dir + "/data/masks", exist_ok=True)

Download dataset


In [ ]:
!wget http://saliencydetection.net/duts/download/DUTS-TR.zip
!wget http://saliencydetection.net/duts/download/DUTS-TE.zip

--2026-05-08 13:09:08--  http://saliencydetection.net/duts/download/DUTS-TR.zip
Resolving saliencydetection.net (saliencydetection.net)... 34.176.215.106
Connecting to saliencydetection.net (saliencydetection.net)|34.176.215.106|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://saliencydetection.net/duts/download/DUTS-TR.zip [following]
--2026-05-08 13:09:08--  https://saliencydetection.net/duts/download/DUTS-TR.zip
Connecting to saliencydetection.net (saliencydetection.net)|34.176.215.106|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 270997309 (258M) [application/zip]
Saving to: ‘DUTS-TR.zip.1’

DUTS-TR.zip.1       100%[===================>] 258.44M  28.0MB/s    in 9.8s    

2026-05-08 13:09:19 (26.3 MB/s) - ‘DUTS-TR.zip.1’ saved [270997309/270997309]

--2026-05-08 13:09:19--  http://saliencydetection.net/duts/download/DUTS-TE.zip
Resolving saliencydetection.net (saliencydetection.net)... 34.176.215.106
Connec

In [ ]:
!unzip DUTS-TR.zip
!unzip DUTS-TE.zip

Archive:  DUTS-TR.zip
replace DUTS-TR/DUTS-TR-Image/ILSVRC2012_test_00000030.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:



img_dir = "DUTS-TR/DUTS-TR-Image"
mask_dir = "DUTS-TR/DUTS-TR-Mask"


if os.path.exists(img_dir) and os.path.exists(mask_dir):
    print("Imazhe te gjetura:", len(os.listdir(img_dir)))
    print("Maska te gjetura:", len(os.listdir(mask_dir)))
else:
    print("GABIM: Folderat nuk u gjeten!")

Preprocessing functions


In [ ]:
def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img


def preprocess_mask(path):
    mask = cv2.imread(path, 0)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    mask = mask / 255.0
    return mask

Datadebugging

In [ ]:
sample_name = os.listdir(img_dir)[0].split('.')[0]

img = preprocess_image(os.path.join(img_dir, sample_name + ".jpg"))
mask = preprocess_mask(os.path.join(mask_dir, sample_name + ".png"))

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Image")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Mask")

plt.show()

Dataset class

In [ ]:
class SODDataset(Dataset):

    def __init__(self, img_dir, mask_dir, img_list):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_list = img_list

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):

        img_name = self.img_list[idx]
        mask_name = img_name.replace(".jpg", ".png")

        image = preprocess_image(os.path.join(self.img_dir, img_name))
        mask = preprocess_mask(os.path.join(self.mask_dir, mask_name))

        image = torch.tensor(image, dtype=torch.float32).permute(2,0,1)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return image, mask

Train Validation Test Split

In [ ]:
all_imgs = sorted(os.listdir(img_dir))

train_list, temp_list = train_test_split(
    all_imgs,
    test_size=0.3,
    random_state=42
)

val_list, test_list = train_test_split(
    temp_list,
    test_size=0.5,
    random_state=42
)

print("Training images:", len(train_list))
print("Validation images:", len(val_list))
print("Test images:", len(test_list))


Create dataset objects

In [ ]:
train_dataset = SODDataset(
    img_dir,
    mask_dir,
    train_list
)

val_dataset = SODDataset(
    img_dir,
    mask_dir,
    val_list
)

test_dataset = SODDataset(
    img_dir,
    mask_dir,
    test_list
)


Create Dataloaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)


Visualize Batch From DataLoader

In [ ]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Mask batch shape:", masks.shape)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(images[0].permute(1,2,0))
plt.title("Sample Training Image")

plt.subplot(1,2,2)
plt.imshow(masks[0].squeeze(), cmap='gray')
plt.title("Sample Training Mask")

plt.show()


Baseline SOD Model

In [ ]:
class BaselineSOD(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(
                32,
                16,
                2,
                stride=2
            ),

            nn.ReLU(),

            nn.ConvTranspose2d(
                16,
                1,
                2,
                stride=2
            )
        )

    def forward(self, x):

        x = self.encoder(x)

        x = self.decoder(x)

        return x


Initialize Baseline Model

In [ ]:
baseline_model = BaselineSOD().to(device)

baseline_criterion = nn.BCEWithLogitsLoss()

baseline_optimizer = optim.Adam(
    baseline_model.parameters(),
    lr=1e-4
)


Train Baseline Model

In [ ]:
baseline_epochs = 5

for epoch in range(baseline_epochs):

    baseline_model.train()

    train_loss = 0

    for images, masks in train_loader:

        images = images.to(device)

        masks = masks.to(device)

        outputs = baseline_model(images)

        loss = baseline_criterion(outputs, masks)

        baseline_optimizer.zero_grad()

        loss.backward()

        baseline_optimizer.step()

        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{baseline_epochs}")
    print(f"Train Loss: {avg_loss:.4f}")



NameError: name 'baseline_model' is not defined

Evaluation Metrics

In [ ]:
def iou(pred, target):

    pred = (pred > 0.4).float()

    target = target.float()

    intersection = (pred * target).sum()

    union = (pred + target - pred * target).sum()

    return (intersection + 1e-6) / (union + 1e-6)



def precision(pred, target):

    pred = (pred > 0.4).float()

    target = target.float()

    tp = (pred * target).sum()

    fp = (pred * (1 - target)).sum()

    return (tp + 1e-6) / (tp + fp + 1e-6)



def recall(pred, target):

    pred = (pred > 0.4).float()

    target = target.float()

    tp = (pred * target).sum()

    fn = ((1 - pred) * target).sum()

    return (tp + 1e-6) / (tp + fn + 1e-6)



def f1(pred, target):

    p = precision(pred, target)

    r = recall(pred, target)

    return (2 * p * r + 1e-6) / (p + r + 1e-6)


Evaluate Baseline Model

In [ ]:
baseline_model.eval()

baseline_iou = 0

baseline_f1 = 0

with torch.no_grad():

    for images, masks in test_loader:

        images = images.to(device)

        masks = masks.to(device)

        outputs = baseline_model(images)

        outputs = torch.sigmoid(outputs)

        baseline_iou += iou(outputs, masks).item()

        baseline_f1 += f1(outputs, masks).item()

baseline_iou /= len(test_loader)

baseline_f1 /= len(test_loader)

print("Baseline Results")
print("----------------")

print(f"IoU: {baseline_iou:.4f}")

print(f"F1 Score: {baseline_f1:.4f}")


Improved SOD Model

In [ ]:
class ImprovedSOD(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = nn.Sequential(

            nn.Conv2d(3, 16, 3, padding=1),

            nn.BatchNorm2d(16),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.enc2 = nn.Sequential(

            nn.Conv2d(16, 32, 3, padding=1),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.enc3 = nn.Sequential(

            nn.Conv2d(32, 64, 3, padding=1),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.MaxPool2d(2)
        )

        self.dec1 = nn.Sequential(

            nn.ConvTranspose2d(
                64,
                32,
                2,
                stride=2
            ),

            nn.ReLU()
        )

        self.dec2 = nn.Sequential(

            nn.ConvTranspose2d(
                32,
                16,
                2,
                stride=2
            ),

            nn.ReLU()
        )

        self.dec3 = nn.ConvTranspose2d(
            16,
            1,
            2,
            stride=2
        )

    def forward(self, x):

        x = self.enc1(x)

        x = self.enc2(x)

        x = self.enc3(x)

        x = self.dec1(x)

        x = self.dec2(x)

        x = self.dec3(x)

        return x


Initialize Model

In [ ]:
model = ImprovedSOD().to(device)

print(model)


Loss Function and Optimizer

In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)


NameError: name 'nn' is not defined

Training and Validation Loop

In [ ]:
epochs = 20

train_losses = []
val_losses = []

for epoch in range(epochs):

    # TRAINING
    model.train()

    train_loss = 0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)

        loss = criterion(outputs, masks)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    train_losses.append(avg_train_loss)

    # VALIDATION
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1}/{epochs}")

    print(f"Train Loss: {avg_train_loss:.4f}")

    print(f"Validation Loss: {avg_val_loss:.4f}")


Plot Training and Validation Loss

In [ ]:
plt.plot(train_losses, label="Train Loss")

plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend()

plt.show()


Evaluate Model on Test Set

In [ ]:
model.eval()

total_iou = 0
total_precision = 0
total_recall = 0
total_f1 = 0

with torch.no_grad():

    for images, masks in test_loader:

        images = images.to(device)

        masks = masks.to(device)

        outputs = model(images)

        outputs = torch.sigmoid(outputs)

        total_iou += iou(outputs, masks).item()

        total_precision += precision(outputs, masks).item()

        total_recall += recall(outputs, masks).item()

        total_f1 += f1(outputs, masks).item()

avg_iou = total_iou / len(test_loader)

avg_precision = total_precision / len(test_loader)

avg_recall = total_recall / len(test_loader)

avg_f1 = total_f1 / len(test_loader)

print("Test Results")
print("------------------")

print(f"IoU: {avg_iou:.4f}")

print(f"Precision: {avg_precision:.4f}")

print(f"Recall: {avg_recall:.4f}")

print(f"F1 Score: {avg_f1:.4f}")


Visualize Predictions

In [ ]:
model.eval()

image, mask = test_dataset[5]

with torch.no_grad():

    prediction = model(
        image.unsqueeze(0).to(device)
    )

    prediction = torch.sigmoid(prediction)

prediction = prediction.squeeze().cpu().numpy()

binary_pred = (prediction > 0.4).astype(np.float32)

image_np = image.permute(1,2,0).numpy()

mask_np = mask.squeeze().numpy()

plt.figure(figsize=(12,5))

plt.subplot(1,3,1)

plt.imshow(image_np)

plt.title("Input Image")

plt.subplot(1,3,2)

plt.imshow(mask_np, cmap='gray')

plt.title("Ground Truth Mask")

plt.subplot(1,3,3)

plt.imshow(binary_pred, cmap='gray')

plt.title("Predicted Mask")

plt.show()


NameError: name 'model' is not defined

Prediction and Overlay Visualization

In [ ]:
model.eval()

# Get one test sample
image, mask = test_dataset[5]

with torch.no_grad():

    prediction = model(
        image.unsqueeze(0).to(device)
    )

    prediction = torch.sigmoid(prediction)

# Convert prediction
prediction = prediction.squeeze().cpu().numpy()

binary_pred = (prediction > 0.4).astype(np.float32)

# Convert tensors to numpy
image_np = image.permute(1,2,0).numpy()

mask_np = mask.squeeze().numpy()

# Create overlay
overlay = image_np.copy()

overlay[:,:,0] = overlay[:,:,0] + binary_pred * 0.5

overlay = np.clip(overlay, 0, 1)

# Plot everything
plt.figure(figsize=(15,5))

plt.subplot(1,4,1)
plt.imshow(image_np)
plt.title("Input Image")

plt.subplot(1,4,2)
plt.imshow(mask_np, cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,4,3)
plt.imshow(binary_pred, cmap='gray')
plt.title("Predicted Mask")

plt.subplot(1,4,4)
plt.imshow(overlay)
plt.title("Overlay")

plt.show()


NameError: name 'model' is not defined

Save Trained Model

In [ ]:
torch.save(
    model.state_dict(),
    "sod_model.pth"
)

print("Model saved successfully!")


Model Comparison

In [ ]:
print("MODEL COMPARISON")
print("-----------------------")

print("Baseline Model")

print(f"IoU: {baseline_iou:.4f}")

print(f"F1 Score: {baseline_f1:.4f}")

print("-----------------------")

print("Improved Model")

print(f"IoU: {avg_iou:.4f}")

print(f"F1 Score: {avg_f1:.4f}")


Save model

In [ ]:
torch.save(model.state_dict(), "sod_model.pth")

Final Result Summary

In [ ]:
print("FINAL PROJECT RESULTS")
print("---------------------")

print("Baseline Model:")
print("IoU: 0.2882")
print("F1: 0.4425")

print("---------------------")

print("Improved Model:")
print("IoU: 0.4547")
print("F1: 0.6231")

print("---------------------")

print("Conclusion:")
print("Improved CNN significantly outperforms baseline model in saliency detection.")


FINAL PROJECT RESULTS
---------------------
Baseline Model:
IoU: 0.2882
F1: 0.4425
---------------------
Improved Model:
IoU: 0.4547
F1: 0.6231
---------------------
Conclusion:
Improved CNN significantly outperforms baseline model in saliency detection.
